In [2]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

from pathlib import Path
WORKDIR = str(Path.cwd().parent) + '/'  # raiz do repositório
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/logs/"+data_hora

In [3]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [4]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [5]:
## get dataset

# select only sensor data from smart watches
#csv_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
#csv_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [6]:
pd.set_option("display.precision", 9)

In [7]:
df_watch = pd.read_csv("activity_recognition_data_watch_join_multi_index.csv",dtype={"value":"float64"})

In [8]:
df_watch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3238804 entries, 0 to 3238803
Data columns (total 9 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_x        int64  
 1   atividade       int64  
 2   tempo           int64  
 3   acelerômetro_x  float64
 4   acelerômetro_Y  float64
 5   acelerômetro_Z  float64
 6   giroscópio_x    float64
 7   giroscópio_Y    float64
 8   giroscópio_Z    float64
dtypes: float64(6), int64(3)
memory usage: 222.4 MB


In [9]:
df_watch.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,0.314944100,-1.02227650,-0.309961600
1,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,0.387382100,-0.61854124,-0.048971802
2,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,0.070998540,-0.20947972,-0.195978300
3,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,0.037975345,0.25497556,-0.156563500
4,1600,0,90426955696641,4.6615110,0.16968904,9.6846950,0.073129070,0.71943086,-0.001034904


In [10]:
df_watch.tail()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
3238799,1650,17,2426184371437252,2.1377227,-7.2114854,8.0654670,-2.8560646,2.0571015,0.85393304
3238800,1650,17,2426184421424592,1.4865000,-13.3023340,5.3552310,-2.9114583,1.8174170,1.28110410
3238801,1650,17,2426184471411932,4.0171714,-13.1467110,5.1182046,-3.0158541,1.5500356,2.05768180
3238802,1650,17,2426184521399272,4.4122143,-15.3182530,3.3704374,-3.2182543,1.5958420,3.01855040
3238803,1650,17,2426184571385262,4.4265795,-13.0126360,2.5755625,-2.4555252,0.8661359,3.35410860


In [11]:
df_watch.shape

(3238804, 9)

In [12]:
df_watch.set_index(["pessoa_x","atividade","tempo"],inplace=True)

In [13]:
df_watch.head()

acelerômetro_x  acelerômetro_Y  \
pessoa_x atividade tempo                                            
1600     0         90426757696641       4.9727570     -0.15831658   
                   90426807196641       3.2537200     -0.19183542   
                   90426856696641       2.8012160     -0.15592238   
                   90426906196641       3.7708676     -1.05135380   
                   90426955696641       4.6615110      0.16968904   

                                   acelerômetro_Z  giroscópio_x  giroscópio_Y  \
pessoa_x atividade tempo                                                        
1600     0         90426757696641       6.6967316   0.314944100   -1.02227650   
                   90426807196641       6.1077580   0.387382100   -0.61854124   
                   90426856696641       5.9976250   0.070998540   -0.20947972   
                   90426906196641       7.7310270   0.037975345    0.25497556   
                   90426955696641       9.6846950   0.073129070    0.71943086   

                                   giroscópio_Z  
pessoa_x atividade tempo                         
1600     0         90426757696641  -0.309961600  
                   90426807196641  -0.048971802  
                   90426856696641  -0.195978300  
                   90426906196641  -0.156563500  
                   90426955696641  -0.001034904

In [14]:
df_phone = pd.read_csv("activity_recognition_data_phone_join_multi_index.csv",dtype={"value":"float64"})

In [15]:
df_phone.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2787819 entries, 0 to 2787818
Data columns (total 9 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_x        int64  
 1   atividade       int64  
 2   tempo           int64  
 3   acelerômetro_x  float64
 4   acelerômetro_Y  float64
 5   acelerômetro_Z  float64
 6   giroscópio_x    float64
 7   giroscópio_Y    float64
 8   giroscópio_Z    float64
dtypes: float64(6), int64(3)
memory usage: 191.4 MB


In [16]:
df_phone.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,252207918580802,-4.33277900,13.3611910,-0.71887210,-0.85321045,0.297225950,0.89018250
1,1600,0,252207968934806,-0.31944275,13.3183590,-0.23202515,-0.87513730,0.015472412,0.16223145
2,1600,0,252208019288809,1.56645200,9.5152740,-0.01777649,-0.72016907,0.388488770,-0.28401184
3,1600,0,252208069642813,-0.32374573,5.2626650,0.32234192,-0.57164000,1.227401700,-0.24166870
4,1600,0,252208119996817,-1.81167600,3.7105103,1.37393190,-0.38049316,1.202835100,-0.21313477


In [17]:
df_phone.tail()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
2787814,1650,17,357733977635000,-8.509504,-3.5206788,-1.8226099,0.43888897,0.171507580,0.16937704
2787815,1650,17,357734017925000,-8.423284,-4.0619530,-1.7603394,0.44421527,0.158724400,0.19707392
2787816,1650,17,357734058325000,-8.327483,-4.1960745,-1.5711329,0.51984906,0.037284255,0.19600865
2787817,1650,17,357734098619000,-8.080796,-4.4188113,-1.4154567,0.54541540,-0.086286420,0.17896444
2787818,1650,17,357734138868000,-8.040081,-4.3661210,-1.4968873,0.47191215,-0.201334980,0.23222765


In [18]:
df_phone.shape

(2787819, 9)

In [19]:
df_phone.set_index(["pessoa_x","atividade","tempo"],inplace=True)

In [20]:
df_phone.head()

acelerômetro_x  acelerômetro_Y  \
pessoa_x atividade tempo                                             
1600     0         252207918580802     -4.33277900      13.3611910   
                   252207968934806     -0.31944275      13.3183590   
                   252208019288809      1.56645200       9.5152740   
                   252208069642813     -0.32374573       5.2626650   
                   252208119996817     -1.81167600       3.7105103   

                                    acelerômetro_Z  giroscópio_x  \
pessoa_x atividade tempo                                           
1600     0         252207918580802     -0.71887210   -0.85321045   
                   252207968934806     -0.23202515   -0.87513730   
                   252208019288809     -0.01777649   -0.72016907   
                   252208069642813      0.32234192   -0.57164000   
                   252208119996817      1.37393190   -0.38049316   

                                    giroscópio_Y  giroscópio_Z  
pessoa_x atividade tempo                                        
1600     0         252207918580802   0.297225950    0.89018250  
                   252207968934806   0.015472412    0.16223145  
                   252208019288809   0.388488770   -0.28401184  
                   252208069642813   1.227401700   -0.24166870  
                   252208119996817   1.202835100   -0.21313477

In [21]:
df_phone.tail()

acelerômetro_x  acelerômetro_Y  \
pessoa_x atividade tempo                                             
1650     17        357733977635000       -8.509504      -3.5206788   
                   357734017925000       -8.423284      -4.0619530   
                   357734058325000       -8.327483      -4.1960745   
                   357734098619000       -8.080796      -4.4188113   
                   357734138868000       -8.040081      -4.3661210   

                                    acelerômetro_Z  giroscópio_x  \
pessoa_x atividade tempo                                           
1650     17        357733977635000      -1.8226099    0.43888897   
                   357734017925000      -1.7603394    0.44421527   
                   357734058325000      -1.5711329    0.51984906   
                   357734098619000      -1.4154567    0.54541540   
                   357734138868000      -1.4968873    0.47191215   

                                    giroscópio_Y  giroscópio_Z  
pessoa_x atividade tempo                                        
1650     17        357733977635000   0.171507580    0.16937704  
                   357734017925000   0.158724400    0.19707392  
                   357734058325000   0.037284255    0.19600865  
                   357734098619000  -0.086286420    0.17896444  
                   357734138868000  -0.201334980    0.23222765

In [22]:
# Select rows with matching indices (inner join)
common_indices = df_phone.index.intersection(df_watch.index)
df1_filtered = df_phone.loc[common_indices]
df2_filtered = df_watch.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1)

In [23]:
result.shape

(0, 12)

In [24]:
result.tail()

,,,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
pessoa_x,atividade,tempo,,,,,,,,,,,,


In [ ]:
result.reset_index(inplace=True)

In [ ]:
result.tail()

In [ ]:
result.columns = ['pessoa_x','atividade','tempo','acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'] 

In [ ]:
result.head()

In [ ]:
result.atividade.value_counts().sort_index()

In [ ]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [ ]:
result["atividade"] = result["atividade"].replace(to_replace=mapping_dict)

In [ ]:
result.head()

In [ ]:
result.tail()

In [ ]:
cols_to_check = ['acelerômetro_Z','giroscópio_Z']

In [ ]:
result[cols_to_check] = result[cols_to_check].replace({';':''}, regex=True)

In [ ]:
result.head()

In [ ]:
result.to_csv(WORKDIR + 'Data/processed/activity_recognition_data_watch_join_multi_index.csv', index=False)